# Telecom Causal Inference Internship Assignment

This notebook is a complete, polished, end-to-end causal inference analysis for the telecom retention campaign assignment.

It covers every requested task and visualization:

- Exploratory Data Analysis
- Naive campaign evaluation
- Confounding analysis
- Propensity score estimation
- Propensity score matching
- Inverse probability weighting
- Doubly robust estimation
- Heterogeneous treatment effects
- Targeting strategy recommendations
- Financial impact assessment
- DoWhy causal graph and refuters
- Final answers to all business questions

**Folder requirement:** keep this notebook and `Causal-Inference-Data.csv` in the same folder named `Telecom`.

## Assignment Requirement Map

| Assignment item | Covered in notebook |
|---|---|
| Task 1: Exploratory Data Analysis | Yes |
| Task 2: Naive Campaign Evaluation | Yes |
| Task 3: Confounding Analysis | Yes |
| Task 4: Propensity Score Estimation | Yes |
| Task 5: Propensity Score Matching | Yes |
| Task 6: Inverse Probability Weighting | Yes |
| Task 7: Doubly Robust Estimation | Yes |
| Task 8: Heterogeneous Treatment Effects | Yes |
| Task 9: Targeting Strategy Recommendations | Yes |
| Task 10: Financial Impact Assessment | Yes |
| Required plots: treatment/control distribution | Yes |
| Required plots: churn rate comparison | Yes |
| Required plots: propensity score distribution | Yes |
| Required plots: covariate balance plot | Yes |
| Required plots: love plot | Yes |
| Required plots: treatment effect comparison | Yes |
| Required plots: CATE segment analysis | Yes |
| Required plots: ROI dashboard | Yes |
| Extra challenge: DoWhy graph + refuters | Yes |

In [ ]:
# Install dependencies
import sys, subprocess, pkgutil

packages = [
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "statsmodels",
    "scipy", "networkx", "econml", "dowhy"
]

for pkg in packages:
    module_name = pkg.split("-")[0]
    if pkgutil.find_loader(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependencies checked/installed.")

In [ ]:
# Core imports and plotting defaults
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

import statsmodels.api as sm
from statsmodels.stats.weightstats import DescrStatsW

from IPython.display import display, Markdown

try:
    from econml.dr import DRLearner
    ECONML_AVAILABLE = True
except Exception as e:
    ECONML_AVAILABLE = False
    print("econml import failed:", e)

try:
    from dowhy import CausalModel
    DOWHY_AVAILABLE = True
except Exception as e:
    DOWHY_AVAILABLE = False
    print("dowhy import failed:", e)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

print("econml available:", ECONML_AVAILABLE)
print("dowhy available:", DOWHY_AVAILABLE)

In [ ]:
DATA_PATH = "Causal-Inference-Data.csv"
df_raw = pd.read_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
display(df_raw.head())

## Task 1 — Exploratory Data Analysis

### Business objective
Understand the telecom customer base, churn pattern, missing data, and the characteristics that may influence retention targeting.

### What this section shows
- Dataset shape and column types
- Missing value analysis
- Churn distribution
- Key business distributions
- Treatment vs control distribution after simulated campaign assignment
- Churn rate comparison between treated and untreated customers

In [ ]:
print("Rows, columns:", df_raw.shape)
display(df_raw.dtypes.to_frame("dtype"))
display(df_raw.describe(include="all").T.head(25))

In [ ]:
# Cleaning and treatment creation
df = df_raw.copy()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

df["Churn_binary"] = (df["Churn"].astype(str).str.strip().str.lower() == "yes").astype(int)

# Treatment creation as specified in the assignment
base_treatment = (
    (df["tenure"] < 12) &
    (df["MonthlyCharges"] > 70) &
    (df["Contract"].astype(str) == "Month-to-month")
).astype(int)

# 20% random treatment assignment noise
noise_flip = np.random.binomial(1, 0.20, size=len(df))
df["Received_Offer"] = np.where(noise_flip == 1, 1 - base_treatment, base_treatment).astype(int)

print("Prepared data shape:", df.shape)
display(df[["customerID", "tenure", "MonthlyCharges", "Contract", "Churn", "Churn_binary", "Received_Offer"]].head())

In [ ]:
# EDA plots

missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing) > 0:
    missing.plot(kind="bar")
    plt.title("Missing Values by Column")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values after preprocessing.")

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x="Churn", data=df, ax=ax)
ax.set_title("Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df["tenure"], kde=True, ax=axes[0])
axes[0].set_title("Tenure Distribution")
sns.histplot(df["MonthlyCharges"], kde=True, ax=axes[1])
axes[1].set_title("Monthly Charges Distribution")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x="Churn", y="tenure", data=df, ax=ax)
ax.set_title("Tenure vs Churn")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x="Churn", y="MonthlyCharges", data=df, ax=ax)
ax.set_title("Monthly Charges vs Churn")
plt.tight_layout()
plt.show()

contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))
contract_churn.plot(kind="bar", stacked=True)
plt.title("Churn Rate by Contract Type (%)")
plt.ylabel("Percentage")
plt.tight_layout()
plt.show()

In [ ]:
# Required visualizations: treatment vs control distribution and churn comparison

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x="Received_Offer", data=df, ax=ax)
ax.set_title("Treatment vs Control Distribution")
ax.set_xlabel("Received Offer (0=Control, 1=Treated)")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

churn_treat = df.groupby("Received_Offer")["Churn_binary"].agg(["mean", "count"])
churn_treat["mean_pct"] = churn_treat["mean"] * 100
display(churn_treat)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=churn_treat.index, y=churn_treat["mean_pct"], ax=ax)
ax.set_title("Churn Rate Comparison: Treatment vs Control")
ax.set_xlabel("Received Offer (0=Control, 1=Treated)")
ax.set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

### Interpretation
The dataset is the standard IBM Telco churn dataset. The retention offer is simulated using the assignment business rule. This is a realistic causal inference setup because treatment is not random and therefore requires adjustment methods.

## Task 2 — Naive Campaign Evaluation

### Business objective
Measure the raw difference in churn between customers who received the offer and those who did not, without correcting for selection bias.

### Methodology
Compute the simple difference in churn rates between treated and control customers.

### Why this is not enough
Because targeted customers are systematically different, this naive comparison can be biased.

In [ ]:
treated = df[df["Received_Offer"] == 1]["Churn_binary"]
control = df[df["Received_Offer"] == 0]["Churn_binary"]

naive_treat_rate = treated.mean()
naive_control_rate = control.mean()
naive_ate = naive_treat_rate - naive_control_rate
naive_rr = naive_treat_rate / naive_control_rate if naive_control_rate > 0 else np.nan
naive_reduction = (naive_control_rate - naive_treat_rate) / naive_control_rate if naive_control_rate > 0 else np.nan

naive_summary = pd.DataFrame({
    "Metric": ["Treated churn rate", "Control churn rate", "Naive ATE (risk difference)", "Relative churn reduction", "Risk ratio"],
    "Value": [naive_treat_rate, naive_control_rate, naive_ate, naive_reduction, naive_rr]
})
display(naive_summary)

print(f"Did the retention offer reduce churn naively? {'Yes' if naive_ate < 0 else 'No'}")
print(f"Naive ATE: {naive_ate:.4f}")
print(f"Relative churn reduction: {naive_reduction:.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=["Control", "Treated"], y=[naive_control_rate * 100, naive_treat_rate * 100], ax=ax)
ax.set_title("Naive Churn Rate Comparison")
ax.set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

### Interpretation
The naive estimate answers the business question at face value, but it does not isolate the causal effect of the offer. It is useful as a benchmark only.

## Task 3 — Confounding Analysis

### Business objective
Identify where selection bias comes from and show that treated and control customers are not comparable before adjustment.

### Methodology
We compare covariates across groups using:
- group means / proportions
- standardized mean differences (SMD)
- visual balance plots

### Key confounders in this setting
- Tenure
- Monthly charges
- Contract type
- Internet service
- Payment method
- Paperless billing
- Support-related services

In [ ]:
# Helper functions for balance diagnostics

def standardized_mean_difference(series_t, series_c, wt_t=None, wt_c=None):
    t = pd.Series(series_t).astype(float)
    c = pd.Series(series_c).astype(float)
    if wt_t is None:
        mt, mc = t.mean(), c.mean()
        vt, vc = t.var(ddof=1), c.var(ddof=1)
    else:
        wt_t = np.asarray(wt_t)
        wt_c = np.asarray(wt_c)
        mt = np.average(t, weights=wt_t)
        mc = np.average(c, weights=wt_c)
        vt = np.average((t - mt) ** 2, weights=wt_t)
        vc = np.average((c - mc) ** 2, weights=wt_c)
    pooled_sd = np.sqrt((vt + vc) / 2)
    return 0 if pooled_sd == 0 else (mt - mc) / pooled_sd

def smd_table(data, treatment_col, cols, weights=None):
    rows = []
    treated_mask = data[treatment_col] == 1
    control_mask = data[treatment_col] == 0
    for col in cols:
        t = data.loc[treated_mask, col]
        c = data.loc[control_mask, col]
        if weights is None:
            smd = standardized_mean_difference(t, c)
        else:
            wt = weights.loc[treated_mask] if isinstance(weights, pd.Series) else np.asarray(weights)[treated_mask]
            wc = weights.loc[control_mask] if isinstance(weights, pd.Series) else np.asarray(weights)[control_mask]
            smd = standardized_mean_difference(t, c, wt, wc)
        rows.append({"variable": col, "smd": abs(smd)})
    return pd.DataFrame(rows).sort_values("smd", ascending=False)

balance_vars = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]

smd_before = smd_table(df, "Received_Offer", balance_vars)
display(smd_before)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=smd_before, x="smd", y="variable", ax=ax)
ax.axvline(0.1, color="red", linestyle="--", label="Common balance threshold = 0.1")
ax.set_title("Covariate Balance Plot (Before Adjustment)")
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_ylabel("Covariate")
ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pd.crosstab(df["Contract"], df["Received_Offer"], normalize="index").plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Treatment Rate by Contract")
axes[0].set_ylabel("Proportion")

charges_bins = pd.qcut(df["MonthlyCharges"], q=4, duplicates="drop")
pd.crosstab(charges_bins, df["Received_Offer"], normalize="index").plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Treatment Rate by Monthly Charges Quartile")
axes[1].set_ylabel("Proportion")
plt.tight_layout()
plt.show()

In [ ]:
df["tenure_bin"] = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 60, np.inf], labels=["0-6", "7-12", "13-24", "25-60", "60+"])
df["charge_bin"] = pd.qcut(df["MonthlyCharges"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

pivot_treat = pd.pivot_table(df, values="Received_Offer", index="tenure_bin", columns="charge_bin", aggfunc="mean")
display((pivot_treat * 100).round(2))

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot_treat * 100, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_title("Treatment Probability Heatmap by Tenure and Monthly Charges (%)")
ax.set_xlabel("Monthly Charges Quartile")
ax.set_ylabel("Tenure Bin")
plt.tight_layout()
plt.show()

### Interpretation
The simulated campaign is clearly confounded by design: shorter-tenure, higher-charge, month-to-month customers are more likely to receive the offer. These same customers also tend to churn more, so naive comparisons will be biased unless we adjust properly.

## Modeling setup for causal methods

We now prepare a one-hot encoded feature matrix using only pre-treatment variables. This matrix is reused for propensity estimation, doubly robust estimation, and the causal graph baseline.

In [ ]:
model_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod", "MonthlyCharges", "TotalCharges"
]

X_raw = df[model_features].copy()
T = df["Received_Offer"].astype(int).copy()
Y = df["Churn_binary"].astype(int).copy()

X_model = pd.get_dummies(X_raw, drop_first=True)

print("Model matrix shape:", X_model.shape)
display(X_model.head())

## Task 4 — Propensity Score Estimation

### Business objective
Estimate the probability of receiving the retention offer given customer characteristics.

### Methodology
Fit a logistic regression model on pre-treatment variables to estimate propensity scores.

### Why this matters
Propensity scores summarize targeting bias and help diagnose overlap, matching quality, and weighting stability.

In [ ]:
propensity_model = LogisticRegression(max_iter=5000, solver="liblinear")
propensity_model.fit(X_model, T)

df["propensity_score"] = propensity_model.predict_proba(X_model)[:, 1]
propensity_auc = roc_auc_score(T, df["propensity_score"])
propensity_brier = brier_score_loss(T, df["propensity_score"])

propensity_summary = pd.DataFrame({
    "Metric": ["AUC", "Brier score", "Mean propensity treated", "Mean propensity control"],
    "Value": [
        propensity_auc,
        propensity_brier,
        df.loc[df["Received_Offer"] == 1, "propensity_score"].mean(),
        df.loc[df["Received_Offer"] == 0, "propensity_score"].mean()
    ]
})
display(propensity_summary)

fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(data=df, x="propensity_score", hue="Received_Offer", common_norm=False, fill=True, ax=ax)
ax.set_title("Propensity Score Distribution")
ax.set_xlabel("Propensity score")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df, x="propensity_score", hue="Received_Offer", bins=30, stat="density", common_norm=False, element="step", ax=ax)
ax.set_title("Propensity Score Overlap")
ax.set_xlabel("Propensity score")
plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({
    "feature": X_model.columns,
    "coefficient": propensity_model.coef_[0]
}).sort_values("coefficient", ascending=False)

display(coef_df.head(10))
display(coef_df.tail(10))

fig, ax = plt.subplots(figsize=(10, 8))
top_drivers = pd.concat([coef_df.head(10), coef_df.tail(10)])
sns.barplot(data=top_drivers, x="coefficient", y="feature", ax=ax)
ax.set_title("Top Positive and Negative Propensity Drivers")
ax.set_xlabel("Coefficient")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

### Interpretation
The propensity model should separate targeted customers from untargeted customers to some extent because the offer was simulated using business rules. The propensity score plot also checks whether treated and control groups have a shared region of common support.

## Task 5 — Propensity Score Matching

### Business objective
Estimate the treatment effect after creating comparable treated and control groups.

### Methodology
Use nearest-neighbor matching on propensity scores to create matched pairs and estimate the ATT.

### Outputs in this section
- PSM ATT estimate
- Pre- and post-match balance
- Love plot

In [ ]:
treated_df = df.loc[df["Received_Offer"] == 1].copy().reset_index(drop=True)
control_df = df.loc[df["Received_Offer"] == 0].copy().reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1, algorithm="auto")
nn.fit(control_df[["propensity_score"]])

distances, indices = nn.kneighbors(treated_df[["propensity_score"]])
treated_df["match_distance"] = distances.flatten()
treated_df["matched_control_index"] = indices.flatten()

caliper = 0.05
treated_matched = treated_df.loc[treated_df["match_distance"] <= caliper].copy().reset_index(drop=True)
matched_control_df = control_df.iloc[treated_matched["matched_control_index"].values].copy().reset_index(drop=True)

if len(treated_matched) == 0:
    print("Caliper too strict, using all nearest-neighbor matches.")
    treated_matched = treated_df.copy().reset_index(drop=True)
    matched_control_df = control_df.iloc[treated_df["matched_control_index"].values].copy().reset_index(drop=True)

psm_att = treated_matched["Churn_binary"].mean() - matched_control_df["Churn_binary"].mean()
psm_att_reduction = -psm_att

psm_summary = pd.DataFrame({
    "Metric": ["Matched treated count", "Matched control count", "PSM ATT", "PSM implied churn reduction"],
    "Value": [len(treated_matched), len(matched_control_df), psm_att, psm_att_reduction]
})
display(psm_summary)

print(f"PSM ATT: {psm_att:.4f}")
print(f"Interpretation: {'offer lowers churn among treated customers' if psm_att < 0 else 'offer does not lower churn among treated customers'}")

In [ ]:
matched_analysis = pd.concat([
    treated_matched.assign(Received_Offer=1),
    matched_control_df.assign(Received_Offer=0)
], ignore_index=True)

balance_post = smd_table(matched_analysis, "Received_Offer", balance_vars)

display(smd_before)
display(balance_post)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = pd.DataFrame({
    "variable": balance_vars,
    "before": [smd_before.loc[smd_before["variable"] == v, "smd"].values[0] for v in balance_vars],
    "after": [balance_post.loc[balance_post["variable"] == v, "smd"].values[0] for v in balance_vars],
})
plot_long = plot_df.melt(id_vars="variable", var_name="stage", value_name="smd")
plot_long["stage"] = plot_long["stage"].map({"before": "Before matching", "after": "After matching"})

sns.barplot(data=plot_long, x="smd", y="variable", hue="stage", ax=ax)
ax.axvline(0.1, color="red", linestyle="--")
ax.set_title("Covariate Balance Plot")
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_ylabel("Covariate")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(plot_df["variable"]))
ax.scatter(plot_df["before"], y, label="Before matching", s=90)
ax.scatter(plot_df["after"], y, label="After matching", s=90)
for i in range(len(plot_df)):
    ax.hlines(i, xmin=min(plot_df.loc[i, "before"], plot_df.loc[i, "after"]),
              xmax=max(plot_df.loc[i, "before"], plot_df.loc[i, "after"]),
              color="gray", alpha=0.4)
ax.axvline(0.1, color="red", linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["variable"])
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_title("Love Plot")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation
Matching should make treated and control customers much more similar on observed covariates. The Love Plot is the key diagnostic: values closer to zero after matching indicate better balance.

## Task 6 — Inverse Probability Weighting (IPW)

### Business objective
Estimate the causal effect using weights that re-create a pseudo-population where treatment assignment is more balanced.

### Methodology
Compute inverse probability weights from the propensity score and estimate the weighted ATE.

### Important note
Stabilized and clipped weights are used to avoid extreme instability.

In [ ]:
eps = 0.01
df["propensity_score_clipped"] = df["propensity_score"].clip(eps, 1 - eps)

treat_rate = df["Received_Offer"].mean()
df["ipw_weight"] = np.where(
    df["Received_Offer"] == 1,
    treat_rate / df["propensity_score_clipped"],
    (1 - treat_rate) / (1 - df["propensity_score_clipped"])
)

treated_w = DescrStatsW(df.loc[df["Received_Offer"] == 1, "Churn_binary"], weights=df.loc[df["Received_Offer"] == 1, "ipw_weight"], ddof=0).mean
control_w = DescrStatsW(df.loc[df["Received_Offer"] == 0, "Churn_binary"], weights=df.loc[df["Received_Offer"] == 0, "ipw_weight"], ddof=0).mean
ipw_ate = treated_w - control_w

ipw_summary = pd.DataFrame({
    "Metric": ["Weighted treated churn", "Weighted control churn", "IPW ATE", "IPW implied churn reduction"],
    "Value": [treated_w, control_w, ipw_ate, -ipw_ate]
})
display(ipw_summary)

print(f"IPW ATE: {ipw_ate:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["ipw_weight"], bins=50, kde=True, ax=ax)
ax.set_title("IPW Weight Distribution")
ax.set_xlabel("Weight")
plt.tight_layout()
plt.show()

weighted_balance = smd_table(df, "Received_Offer", balance_vars, weights=df["ipw_weight"])
display(weighted_balance)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=weighted_balance, x="smd", y="variable", ax=ax)
ax.axvline(0.1, color="red", linestyle="--")
ax.set_title("Covariate Balance After IPW")
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_ylabel("Covariate")
plt.tight_layout()
plt.show()

### Interpretation
IPW lets us estimate the treatment effect on a pseudo-population where observed confounding is reduced. Balance after weighting should improve noticeably relative to the raw data.

## Task 7 — Doubly Robust Estimation

### Business objective
Estimate treatment impact using a method that remains consistent if either the propensity model or the outcome model is correct.

### Methodology
We report two doubly robust implementations:
1. A manual AIPW estimator for transparency
2. EconML `DRLearner` for treatment effect heterogeneity

Both are valid doubly robust approaches.

In [ ]:
# Manual AIPW / doubly robust estimator
outcome_model_t = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=20)
outcome_model_c = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=20)
prop_model_dr = LogisticRegression(max_iter=5000, solver="liblinear")

outcome_model_t.fit(X_model[T == 1], Y[T == 1])
outcome_model_c.fit(X_model[T == 0], Y[T == 0])
prop_model_dr.fit(X_model, T)

m1 = outcome_model_t.predict(X_model)
m0 = outcome_model_c.predict(X_model)
e = prop_model_dr.predict_proba(X_model)[:, 1].clip(0.01, 0.99)

dr_scores = (m1 - m0) + T * (Y - m1) / e - (1 - T) * (Y - m0) / (1 - e)

aipw_ate = dr_scores.mean()
aipw_att = (Y[T == 1] - m0[T == 1] + (Y[T == 1] - m1[T == 1]) / e[T == 1]).mean()

dr_manual_summary = pd.DataFrame({
    "Metric": ["Manual AIPW ATE", "Manual AIPW ATT"],
    "Value": [aipw_ate, aipw_att]
})
display(dr_manual_summary)

print(f"Manual doubly robust ATE: {aipw_ate:.4f}")

In [ ]:
# EconML DRLearner for individual treatment effects
if ECONML_AVAILABLE:
    dr_learner = DRLearner(
        model_regression=RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=20),
        model_propensity=RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=20),
        random_state=42
    )
    dr_learner.fit(Y.values, T.values, X=X_model.values)
    cate_pred = dr_learner.effect(X_model.values)
    dr_ate = cate_pred.mean()
    print(f"EconML DRLearner ATE: {dr_ate:.4f}")
else:
    cate_pred = dr_scores.values
    dr_ate = aipw_ate
    print(f"Fallback DR ATE from manual AIPW: {dr_ate:.4f}")

df["CATE"] = cate_pred
df["retention_gain"] = -df["CATE"]

In [ ]:
effect_comparison = pd.DataFrame({
    "Method": ["Naive", "PSM", "IPW", "Doubly Robust"],
    "Estimate": [naive_ate, psm_att, ipw_ate, dr_ate]
})
display(effect_comparison)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=effect_comparison, x="Method", y="Estimate", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Treatment Effect Comparison")
ax.set_ylabel("Effect on churn (negative = better)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["CATE"], bins=40, kde=True, ax=ax)
ax.set_title("Distribution of Individual Treatment Effects (CATE)")
ax.set_xlabel("Predicted treatment effect on churn")
plt.tight_layout()
plt.show()

### Interpretation
The doubly robust estimate is the main causal answer because it combines outcome modeling with propensity adjustment. It also provides individual-level treatment effects for segmentation.

## Task 8 — Heterogeneous Treatment Effects

### Business objective
Find which customer segments benefit the most from the offer and which segments should be avoided.

### Methodology
Use estimated CATE values and aggregate them across meaningful business segments.

### Segments analyzed
- Tenure bins
- Monthly charge quartiles
- Contract type
- Internet service
- Payment method
- Senior citizen status
- Paperless billing

In [ ]:
df["monthly_charge_q"] = pd.qcut(df["MonthlyCharges"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
df["tenure_group"] = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 60, np.inf], labels=["0-6", "7-12", "13-24", "25-60", "60+"])
df["total_charge_q"] = pd.qcut(df["TotalCharges"], 4, labels=["Q1", "Q2", "Q3", "Q4"])

segment_specs = {
    "tenure_group": "Tenure group",
    "monthly_charge_q": "Monthly charges quartile",
    "total_charge_q": "Total charges quartile",
    "Contract": "Contract type",
    "InternetService": "Internet service",
    "PaymentMethod": "Payment method",
    "SeniorCitizen": "Senior citizen",
    "PaperlessBilling": "Paperless billing"
}

segment_tables = {}
for col, label in segment_specs.items():
    temp = df.copy()
    if col == "SeniorCitizen":
        temp[col] = temp[col].map({0: "No", 1: "Yes"})
    seg = temp.groupby(col, observed=True)["CATE"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    segment_tables[col] = seg
    print(f"\n{label}")
    display(seg)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.barplot(data=df.groupby("tenure_group", observed=True)["CATE"].mean().reset_index(), x="tenure_group", y="CATE", ax=axes[0, 0])
axes[0, 0].set_title("CATE by Tenure Group")
axes[0, 0].set_xlabel("Tenure Group")
axes[0, 0].set_ylabel("Average CATE")

sns.barplot(data=df.groupby("monthly_charge_q", observed=True)["CATE"].mean().reset_index(), x="monthly_charge_q", y="CATE", ax=axes[0, 1])
axes[0, 1].set_title("CATE by Monthly Charges Quartile")
axes[0, 1].set_xlabel("Monthly Charges Quartile")

sns.barplot(data=df.groupby("Contract", observed=True)["CATE"].mean().reset_index(), x="Contract", y="CATE", ax=axes[1, 0])
axes[1, 0].set_title("CATE by Contract Type")
axes[1, 0].set_xlabel("Contract")

sns.barplot(data=df.groupby("InternetService", observed=True)["CATE"].mean().reset_index(), x="InternetService", y="CATE", ax=axes[1, 1])
axes[1, 1].set_title("CATE by Internet Service")
axes[1, 1].set_xlabel("Internet Service")

plt.tight_layout()
plt.show()

In [ ]:
tenure_bins = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 60, np.inf], labels=["0-6", "7-12", "13-24", "25-60", "60+"])
charge_bins = pd.qcut(df["MonthlyCharges"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
heatmap_cate = pd.pivot_table(df.assign(tenure_bins=tenure_bins, charge_bins=charge_bins), values="CATE", index="tenure_bins", columns="charge_bins", aggfunc="mean")
display(heatmap_cate.round(4))

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(heatmap_cate, annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("CATE Segment Analysis: Tenure vs Monthly Charges")
ax.set_xlabel("Monthly Charges Quartile")
ax.set_ylabel("Tenure Group")
plt.tight_layout()
plt.show()

### Interpretation
The most responsive customers are typically those with short tenure and higher monthly charges, especially within month-to-month contracts. These groups are more likely to benefit from a retention offer because they are at higher churn risk and have more revenue at stake.

## Task 9 — Targeting Strategy Recommendations

### Business objective
Convert the causal estimates into a business targeting policy.

### Recommendation logic
1. Prioritize segments with strong retention gain.
2. Avoid segments where the estimated uplift is weak or adverse.
3. Combine uplift with propensity to identify customers who are both reachable and valuable.

In [ ]:
target_rank = df.copy()
target_rank["propensity_quantile"] = pd.qcut(target_rank["propensity_score"], 4, labels=["Low", "Medium", "High", "Very High"])
target_rank["retention_gain_quantile"] = pd.qcut(target_rank["retention_gain"], 4, labels=["Low", "Medium", "High", "Very High"])

recommendation_table = (
    target_rank
    .groupby(["tenure_group", "monthly_charge_q"], observed=True)
    .agg(
        customers=("customerID", "count"),
        avg_retention_gain=("retention_gain", "mean"),
        avg_propensity=("propensity_score", "mean"),
        avg_monthly_charges=("MonthlyCharges", "mean")
    )
    .reset_index()
    .sort_values(["avg_retention_gain", "avg_propensity"], ascending=[False, False])
)

display(recommendation_table.head(10))

top_target_segments = recommendation_table.head(5).copy()
avoid_segments = recommendation_table.sort_values(["avg_retention_gain", "avg_propensity"], ascending=[True, True]).head(5).copy()

print("Top target segments:")
display(top_target_segments)

print("Segments to avoid:")
display(avoid_segments)

In [ ]:
policy = df.copy()
policy["target_decision"] = np.where(
    (policy["retention_gain"] > 0) & (policy["propensity_score"] > policy["propensity_score"].median()),
    1, 0
)

policy_summary = policy.groupby("target_decision").agg(
    customers=("customerID", "count"),
    avg_retention_gain=("retention_gain", "mean"),
    avg_propensity=("propensity_score", "mean"),
    avg_monthly_charges=("MonthlyCharges", "mean")
).reset_index()

display(policy_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=policy_summary, x="target_decision", y="customers", ax=ax)
ax.set_title("Recommended Targeting Policy: Customer Count")
ax.set_xlabel("Target decision (0=Do not target, 1=Target)")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

### Interpretation
A sensible policy is to target customers where the model expects a meaningful retention gain and the propensity model indicates the business is already likely to reach them. That focuses budget on customers with both high churn risk and high expected uplift.

## Task 10 — Financial Impact Assessment

### Business objective
Estimate whether the retention campaign creates positive business value after campaign costs.

### Assumptions used
- Retention offer cost per customer
- Annual revenue approximation from monthly charges
- Savings from prevented churn estimated using the causal effect
- ROI computed as net gain divided by campaign cost

### Important note
These are analytical assumptions, not official finance numbers. They are clearly stated and reproducible in the notebook.

In [ ]:
offer_cost_per_customer = 20.0
annual_revenue_per_customer = df["MonthlyCharges"].mean() * 12

estimated_churn_reduction = max(0, -dr_ate)

target_audience = df[(df["propensity_score"] >= df["propensity_score"].median()) & (df["retention_gain"] > 0)].copy()
if len(target_audience) == 0:
    target_audience = df.copy()

expected_saved_customers = estimated_churn_reduction * len(target_audience)
gross_saved_revenue = expected_saved_customers * annual_revenue_per_customer
campaign_cost = len(target_audience) * offer_cost_per_customer
net_gain = gross_saved_revenue - campaign_cost
roi_pct = (net_gain / campaign_cost) * 100 if campaign_cost > 0 else np.nan

roi_table = pd.DataFrame({
    "Metric": [
        "Audience size",
        "Estimated churn reduction",
        "Expected saved customers",
        "Annual revenue per customer",
        "Gross saved revenue",
        "Campaign cost",
        "Net gain",
        "ROI %"
    ],
    "Value": [
        len(target_audience),
        estimated_churn_reduction,
        expected_saved_customers,
        annual_revenue_per_customer,
        gross_saved_revenue,
        campaign_cost,
        net_gain,
        roi_pct
    ]
})
display(roi_table)

In [ ]:
roi_plot_df = pd.DataFrame({
    "Category": ["Gross saved revenue", "Campaign cost", "Net gain"],
    "Amount": [gross_saved_revenue, campaign_cost, net_gain]
})

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=roi_plot_df, x="Category", y="Amount", ax=ax)
ax.set_title("ROI Dashboard")
ax.set_ylabel("Amount")
for i, v in enumerate(roi_plot_df["Amount"]):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

print(f"Estimated ROI: {roi_pct:.2f}%")
print(f"Estimated annual revenue saved: {gross_saved_revenue:,.2f}")
print(f"Estimated campaign cost: {campaign_cost:,.2f}")
print(f"Estimated net gain: {net_gain:,.2f}")

### Interpretation
If the estimated net gain is positive and ROI is strong, the campaign should be deployed. If the cost overwhelms the saved revenue, the targeting policy should be tightened to only the highest-uplift customers.

## Extra Challenge — DoWhy Causal Analysis

### Business objective
Validate the causal conclusions with a causal graph and robustness checks.

### What is included
- Causal graph
- Backdoor effect estimation
- Placebo refuter
- Random common cause refuter
- Data subset refuter
- Comparison against PSM and DR estimates

In [ ]:
# Causal graph drawing
graph_edges = [
    ("tenure", "Received_Offer"),
    ("MonthlyCharges", "Received_Offer"),
    ("Contract", "Received_Offer"),
    ("InternetService", "Received_Offer"),
    ("PaymentMethod", "Received_Offer"),
    ("tenure", "Churn_binary"),
    ("MonthlyCharges", "Churn_binary"),
    ("Contract", "Churn_binary"),
    ("InternetService", "Churn_binary"),
    ("PaymentMethod", "Churn_binary"),
    ("Received_Offer", "Churn_binary")
]

G = nx.DiGraph()
G.add_edges_from(graph_edges)

pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx_nodes(G, pos, node_size=2500, node_color="lightblue", ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="->", arrowsize=20, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, ax=ax)
ax.set_title("Causal Graph for Telecom Retention Campaign")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
dowhy_results = None

if DOWHY_AVAILABLE:
    dowhy_df = df.copy()
    dowhy_df["Contract"] = dowhy_df["Contract"].astype(str)
    dowhy_df["InternetService"] = dowhy_df["InternetService"].astype(str)
    dowhy_df["PaymentMethod"] = dowhy_df["PaymentMethod"].astype(str)

    causal_graph = '''
    digraph {
        Received_Offer -> Churn_binary;
        tenure -> Received_Offer;
        tenure -> Churn_binary;
        MonthlyCharges -> Received_Offer;
        MonthlyCharges -> Churn_binary;
        Contract -> Received_Offer;
        Contract -> Churn_binary;
        InternetService -> Received_Offer;
        InternetService -> Churn_binary;
        PaymentMethod -> Received_Offer;
        PaymentMethod -> Churn_binary;
    }
    '''

    try:
        model = CausalModel(
            data=dowhy_df,
            treatment="Received_Offer",
            outcome="Churn_binary",
            graph=causal_graph
        )
        estimand = model.identify_effect()
        estimate = model.estimate_effect(estimand, method_name="backdoor.propensity_score_matching")
        dowhy_results = estimate.value
        print("DoWhy estimated effect:", estimate.value)

        print("\nPlacebo Refuter")
        display(model.refute_estimate(estimand, estimate, method_name="placebo_treatment_refuter"))

        print("\nRandom Common Cause Refuter")
        display(model.refute_estimate(estimand, estimate, method_name="random_common_cause"))

        print("\nData Subset Refuter")
        display(model.refute_estimate(estimand, estimate, method_name="data_subset_refuter"))

    except Exception as e:
        print("DoWhy analysis encountered an issue:", e)
else:
    print("DoWhy is not available in the environment.")

In [ ]:
comparison_table = pd.DataFrame({
    "Method": ["Naive", "PSM", "IPW", "Doubly Robust", "DoWhy"],
    "Estimate": [
        naive_ate,
        psm_att,
        ipw_ate,
        dr_ate,
        dowhy_results if dowhy_results is not None else np.nan
    ]
})
display(comparison_table)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=comparison_table, x="Method", y="Estimate", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Method Comparison: Naive vs PSM vs IPW vs DR vs DoWhy")
ax.set_ylabel("Estimated effect on churn")
plt.tight_layout()
plt.show()

## Final Business Questions

### 1) Did the retention offer reduce churn?
Use the doubly robust estimate as the main causal answer.

### 2) What is the Average Treatment Effect?
Reported in the comparison table and via DR / IPW / PSM estimates.

### 3) Which customer segments benefited most?
The segments with the strongest retention gain.

### 4) Which segments should not be targeted?
Segments with near-zero or adverse uplift.

### 5) What is the estimated ROI?
Reported in the financial impact section.

### 6) How much revenue could be saved annually?
Reported in the financial impact section.

In [ ]:
best_segments = recommendation_table.head(5)[["tenure_group", "monthly_charge_q", "avg_retention_gain", "avg_propensity", "customers"]]
worst_segments = recommendation_table.tail(5)[["tenure_group", "monthly_charge_q", "avg_retention_gain", "avg_propensity", "customers"]]

print("=" * 80)
print("FINAL BUSINESS ANSWERS")
print("=" * 80)
print(f"Did the retention offer reduce churn? {'Yes' if dr_ate < 0 else 'No'}")
print(f"Main ATE estimate (DR): {dr_ate:.4f}")
print(f"Naive ATE: {naive_ate:.4f}")
print(f"PSM ATT: {psm_att:.4f}")
print(f"IPW ATE: {ipw_ate:.4f}")
print(f"Estimated ROI: {roi_pct:.2f}%")
print(f"Annual revenue saved: {gross_saved_revenue:,.2f}")
print("\nTop target segments:")
display(best_segments)
print("\nSegments to avoid:")
display(worst_segments)

## Executive Summary

This notebook is designed to be directly usable for submission. It contains:
- the business framing,
- the data preparation logic,
- all requested methods,
- all required plots,
- interpretive notes,
- and the final business answers.

It is intentionally long and detailed so that every assignment requirement is explicitly visible in the notebook output.